# Modellazione - Red Flag Detection

Questo notebook implementa la prima fase (**Obiettivo 1**) della modellazione dell'Intelligenza Artificiale per il riconoscimento del rischio suicidario:
- **Data Splitting**: Ripartizione del dataset processato in set di Training (80%), Validation (10%) e Test (10%).
- **Stratificazione**: Mantenimento rigoroso del bilanciamento naturale delle classi individuato nella Data Preparation.
- **Baseline Model**: Protocollo di testing di Benchmark Classico (implementazione a seguire nei prossimi commit).

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
# Per ora ignoriamo i warning seaborn sui target text
import warnings; warnings.filterwarnings('ignore')

df = pd.read_csv('../data/processed/red_flag_cleaned.csv')
print(f"Totale righe Dataset Filtrato: {len(df)}")
display(df['class'].value_counts(normalize=True))

### 1. Data Splitting (80 / 10 / 10)
Utilizziamo la stratificazione (`stratify=y`) garantendo che la proporzione al 50.x% rimanga inalterata nei tre sotto-insiemi matematici.

In [ ]:
# Separazione 80% Train, 20% Temporaneo (da dividere in Val/Test)
train_df, temp_df = train_test_split(df, test_size=0.20, stratify=df['class'], random_state=42)

# Dividiamo il restante 20% in due parti uguali (10% Val, 10% Test)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df['class'], random_state=42)

print(f"Dimensione Training Set:   {len(train_df)} righe ({len(train_df)/len(df)*100:.1f}%)")
print(f"Dimensione Validation Set: {len(val_df)} righe ({len(val_df)/len(df)*100:.1f}%)")
print(f"Dimensione Test Set:       {len(test_df)} righe ({len(test_df)/len(df)*100:.1f}%)")

### 2. Validazione del Bilanciamento e Rappresentazione

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

sns.barplot(x=['Train', 'Validation', 'Test'], y=[len(train_df), len(val_df), len(test_df)], ax=axes[0], palette='viridis')
axes[0].set_title('Suddivisione dei Dati (80/10/10)')
axes[0].set_ylabel('Numero Istanze')

balance_data = pd.DataFrame({
    'Split': ['Train', 'Train', 'Val', 'Val', 'Test', 'Test'],
    'Class': ['non-suicide', 'suicide'] * 3,
    'Count': [
        sum(train_df['class']=='non-suicide'), sum(train_df['class']=='suicide'),
        sum(val_df['class']=='non-suicide'), sum(val_df['class']=='suicide'),
        sum(test_df['class']=='non-suicide'), sum(test_df['class']=='suicide')
    ]
})
sns.barplot(data=balance_data, x='Split', y='Count', hue='Class', ax=axes[1], palette='Set2')
axes[1].set_title('Mantenimento Bilanciamento Classi (Stratified)')
plt.tight_layout()
plt.show()

### 3. Esportazione File per Modellazione Seguente

In [ ]:
train_df.to_csv('../data/processed/train.csv', index=False)
val_df.to_csv('../data/processed/val.csv', index=False)
test_df.to_csv('../data/processed/test.csv', index=False)
print("I tre dataset separati sono stati salvati in `data/processed/` per lo step baseline.")

## 4. Addestramento del Modello Baseline

In questa fase definiamo una baseline solida e interpretabile da utilizzare come benchmark nel corso del ciclo di esperimenti.\
La scelta ricade su un'architettura ibrida basata sulla vettorializzazione **TF-IDF** (Term Frequency - Inverse Document Frequency) accoppiata a un classificatore a **Regressione Logistica** (Logistic Regression).

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
import joblib
import os

# Assicuriamoci che non ci siano valori NaN nel testo
train_text = train_df['text'].fillna('')
val_text = val_df['text'].fillna('')

# 1. Vettorizzazione TF-IDF
vectorizer = TfidfVectorizer(max_features=10000)
X_train = vectorizer.fit_transform(train_text)
X_val = vectorizer.transform(val_text)

y_train = train_df['class']
y_val = val_df['class']

print(f"Dimensioni matrice Training (Testi processati, Feature): {X_train.shape}")

In [ ]:
# 2. Inizializzazione e Addestramento Logistic Regression
clf = LogisticRegression(random_state=42, max_iter=1000)
clf.fit(X_train, y_train)
print("Addestramento terminato.")

In [ ]:
# 3. Test Rapido sul Set di Validazione
y_pred_val = clf.predict(X_val)

print("--- Report Classificazione (Validation) ---")
print(classification_report(y_val, y_pred_val))

cm = confusion_matrix(y_val, y_pred_val, labels=['non-suicide', 'suicide'])
print("\nMatrice di Confusione:")
print(cm)

In [ ]:
os.makedirs('../weights', exist_ok=True)
joblib.dump(vectorizer, '../weights/tfidf_vectorizer_baseline.pkl')
joblib.dump(clf, '../weights/logreg_baseline.pkl')
print("Oggetti esportati in `/weights` per la fase di ottimizzazione delle soglie.")

## 4. Addestramento del Modello Baseline

In questa fase definiamo una baseline solida e interpretabile da utilizzare come benchmark nel corso del ciclo di esperimenti.\
La scelta ricade su un'architettura ibrida basata sulla vettorializzazione **TF-IDF** (Term Frequency - Inverse Document Frequency) accoppiata a un classificatore a **Regressione Logistica** (Logistic Regression).

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
import joblib
import os

# Assicuriamoci che non ci siano valori NaN nel testo
train_text = train_df['text'].fillna('')
val_text = val_df['text'].fillna('')

# 1. Vettorizzazione TF-IDF
vectorizer = TfidfVectorizer(max_features=10000)
X_train = vectorizer.fit_transform(train_text)
X_val = vectorizer.transform(val_text)

y_train = train_df['class']
y_val = val_df['class']

print(f"Dimensioni matrice Training (Testi processati, Feature): {X_train.shape}")

In [ ]:
# 2. Inizializzazione e Addestramento Logistic Regression
clf = LogisticRegression(random_state=42, max_iter=1000)
clf.fit(X_train, y_train)
print("Addestramento terminato.")

In [ ]:
# 3. Test Rapido sul Set di Validazione
y_pred_val = clf.predict(X_val)

print("--- Report Classificazione (Validation) ---")
print(classification_report(y_val, y_pred_val))

cm = confusion_matrix(y_val, y_pred_val, labels=['non-suicide', 'suicide'])
print("\nMatrice di Confusione:")
print(cm)

In [ ]:
os.makedirs('../weights', exist_ok=True)
joblib.dump(vectorizer, '../weights/tfidf_vectorizer_baseline.pkl')
joblib.dump(clf, '../weights/logreg_baseline.pkl')
print("Oggetti esportati in `/weights` per la fase di ottimizzazione delle soglie.")